In [1]:
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from prettytable import PrettyTable

d:\TextImageDiffussion\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Retrieval Metrics functions
def compute_mrr(relevances):
    """Compute Mean Reciprocal Rank for a single ranked list of binary relevances."""
    for idx, rel in enumerate(relevances, start=1):
        if rel:
            return 1.0 / idx
    return 0.0

In [3]:
def average_precision(relevances):
    """Compute Average Precision (AP) for a single ranked list."""
    ap = 0.0
    relevant_count = 0
    for idx, rel in enumerate(relevances, start=1):
        if rel:
            relevant_count += 1
            ap += relevant_count / idx
    return ap / relevant_count if relevant_count > 0 else 0.0

In [4]:
def ndcg(relevances):
    """Compute Normalized Discounted Cumulative Gain (NDCG) for a ranked list."""
    def dcg(scores):
        return sum([(2**score - 1) / np.log2(idx + 2) for idx, score in enumerate(scores)])
    dcg_val = dcg(relevances)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg_val = dcg(ideal_relevances)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

In [5]:
# --- List of embedding model names ---
model_names = [
    "all-mpnet-base-v2",
    "multi-qa-mpnet-base-dot-v1",
    "all-distilroberta-v1",
    "all-MiniLM-L12-v2",
    "multi-qa-distilbert-cos-v1",
    "all-MiniLM-L6-v2",
    "multi-qa-MiniLM-L6-cos-v1",
    "paraphrase-multilingual-mpnet-base-v2",
    "paraphrase-albert-small-v2",
    "paraphrase-multilingual-MiniLM-L12-v2",
    "paraphrase-MiniLM-L3-v2",
    "distiluse-base-multilingual-cased-v1",
    "distiluse-base-multilingual-cased-v2"
]

In [6]:
# --- Load extrinsic dataset: ag_news ---
print("Loading ag_news dataset for extrinsic evaluation...")
ag_news = load_dataset("ag_news")
# For speed, we use a small random subset
train_ag = ag_news["train"].shuffle(seed=42).select(range(500))
test_ag = ag_news["test"].shuffle(seed=42).select(range(100))
train_ag_texts = train_ag["text"]
train_ag_labels = train_ag["label"]
test_ag_texts = test_ag["text"]
test_ag_labels = test_ag["label"]

Loading ag_news dataset for extrinsic evaluation...


In [7]:
# --- Load intrinsic dataset: STS Benchmark (multilingual version in English) ---
print("Loading STS Benchmark dataset for intrinsic evaluation...")
stsb = load_dataset("stsb_multi_mt", "en", split="test")
# Use a subset for quick evaluation
stsb = stsb.shuffle(seed=42).select(range(200))
stsb_sentences1 = stsb["sentence1"]
stsb_sentences2 = stsb["sentence2"]
stsb_scores = stsb["similarity_score"]  # scores typically range from 0 to 5
# Create binary ground truth: similar if score >= 2.5, else not similar
stsb_binary = [1 if score >= 2.5 else 0 for score in stsb_scores]

Loading STS Benchmark dataset for intrinsic evaluation...


In [8]:
import gc
import torch

In [9]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print("Using GPU:", torch.cuda.get_device_name(0))

CUDA is available!
Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [10]:
# --- Storage for all results ---
results = []

In [11]:
# --- Evaluate each model ---
for model_name in model_names:
    print(f"\nEvaluating model: {model_name}")
    try:
        model = SentenceTransformer(model_name,device="cuda")
        if torch.cuda.is_available():
           device = "cuda" 
        print(device)
    except Exception as e:
        print(f"Error loading model {model_name}: {e}")
        continue

    # ===== Extrinsic Evaluation on ag_news =====
    # Measure time for creating embeddings for the classification/retrieval task.
    start_time = time.time()
    train_ag_embeddings = model.encode(train_ag_texts, show_progress_bar=False, convert_to_tensor=True)
    test_ag_embeddings = model.encode(test_ag_texts, show_progress_bar=False, convert_to_tensor=True)
    extrinsic_embed_time = time.time() - start_time

    # ---- Classification with Logistic Regression ----
    X_train = train_ag_embeddings.cpu().numpy()
    X_test = test_ag_embeddings.cpu().numpy()
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, train_ag_labels)
    pred_ag = clf.predict(X_test)
    ext_accuracy = accuracy_score(test_ag_labels, pred_ag)
    ext_precision = precision_score(test_ag_labels, pred_ag, average="macro", zero_division=0)
    ext_recall = recall_score(test_ag_labels, pred_ag, average="macro", zero_division=0)

    # ---- Retrieval Evaluation (simulate a retrieval task)
    # For each test example, rank all training examples by cosine similarity.
    mrr_list, map_list, ndcg_list = [], [], []
    for i, test_emb in enumerate(test_ag_embeddings):
        # Compute cosine similarities with all training embeddings.
        cos_scores = util.cos_sim(test_emb, train_ag_embeddings)[0]
        sorted_indices = np.argsort(cos_scores.cpu().numpy())[::-1]
        # Define relevant documents: those with the same label.
        relevances = [1 if train_ag_labels[idx] == test_ag_labels[i] else 0 for idx in sorted_indices]
        mrr_list.append(compute_mrr(relevances))
        map_list.append(average_precision(relevances))
        ndcg_list.append(ndcg(relevances))
    ret_mrr = np.mean(mrr_list)
    ret_map = np.mean(map_list)
    ret_ndcg = np.mean(ndcg_list)

    # ===== Intrinsic Evaluation on STS Benchmark =====
    # Measure time for creating embeddings for the intrinsic task.
    start_time = time.time()
    stsb_emb1 = model.encode(stsb_sentences1, show_progress_bar=False, convert_to_tensor=True)
    stsb_emb2 = model.encode(stsb_sentences2, show_progress_bar=False, convert_to_tensor=True)
    intrinsic_embed_time = time.time() - start_time

    # For overall embedding time, we sum extrinsic + intrinsic times.
    total_embed_time = extrinsic_embed_time + intrinsic_embed_time

    # Compute cosine similarity for each sentence pair.
    stsb_cosine_sim = [
        float(util.cos_sim(stsb_emb1[i], stsb_emb2[i]))
        for i in range(len(stsb_emb1))
    ]
    # Using a fixed threshold (e.g. 0.7) to decide if a pair is similar.
    stsb_pred_binary = [1 if sim >= 0.7 else 0 for sim in stsb_cosine_sim]
    int_accuracy = accuracy_score(stsb_binary, stsb_pred_binary)
    int_precision = precision_score(stsb_binary, stsb_pred_binary, zero_division=0)
    int_recall = recall_score(stsb_binary, stsb_pred_binary, zero_division=0)

    # --- Store all metrics for this model ---
    results.append({
        "Model": model_name,
        "Ext_Acc": ext_accuracy,
        "Ext_Prec": ext_precision,
        "Ext_Rec": ext_recall,
        "Ret_MRR": ret_mrr,
        "Ret_MAP": ret_map,
        "Ret_NDCG": ret_ndcg,
        "Int_Acc": int_accuracy,
        "Int_Prec": int_precision,
        "Int_Rec": int_recall,
        "Embed_Time": total_embed_time
    })

    del model  # Delete the model
    gc.collect()  # Force garbage collection
    if torch.cuda.is_available():
       torch.cuda.empty_cache() 



Evaluating model: all-mpnet-base-v2
cuda

Evaluating model: multi-qa-mpnet-base-dot-v1
cuda

Evaluating model: all-distilroberta-v1
cuda

Evaluating model: all-MiniLM-L12-v2
cuda

Evaluating model: multi-qa-distilbert-cos-v1
cuda

Evaluating model: all-MiniLM-L6-v2
cuda

Evaluating model: multi-qa-MiniLM-L6-cos-v1
cuda

Evaluating model: paraphrase-multilingual-mpnet-base-v2
Error loading model paraphrase-multilingual-mpnet-base-v2: 

Evaluating model: paraphrase-albert-small-v2
cuda

Evaluating model: paraphrase-multilingual-MiniLM-L12-v2
cuda

Evaluating model: paraphrase-MiniLM-L3-v2
cuda

Evaluating model: distiluse-base-multilingual-cased-v1
cuda

Evaluating model: distiluse-base-multilingual-cased-v2
cuda


In [12]:
table = PrettyTable()
table.field_names = ["Model", "Ext_Acc", "Ext_Prec", "Ext_Rec", "Ret_MRR", "Ret_MAP", "Ret_NDCG", "Int_Acc", "Int_Prec", "Int_Rec", "Embed_Time(s)"]

for res in results:
    table.add_row([
        res["Model"],
        f"{res['Ext_Acc']:.4f}",
        f"{res['Ext_Prec']:.4f}",
        f"{res['Ext_Rec']:.4f}",
        f"{res['Ret_MRR']:.4f}",
        f"{res['Ret_MAP']:.4f}",
        f"{res['Ret_NDCG']:.4f}",
        f"{res['Int_Acc']:.4f}",
        f"{res['Int_Prec']:.4f}",
        f"{res['Int_Rec']:.4f}",
        f"{res['Embed_Time']:.2f}"
    ])

print("\nEvaluation Results:")
print(table)

# --- Determine the best model based on extrinsic classification accuracy ---
if results:
    best_model = max(results, key=lambda x: x["Ext_Acc"])["Model"]
    print(f"\nBest model based on extrinsic accuracy: {best_model}")
else:
    print("No results to evaluate.")


Evaluation Results:
+---------------------------------------+---------+----------+---------+---------+---------+----------+---------+----------+---------+---------------+
|                 Model                 | Ext_Acc | Ext_Prec | Ext_Rec | Ret_MRR | Ret_MAP | Ret_NDCG | Int_Acc | Int_Prec | Int_Rec | Embed_Time(s) |
+---------------------------------------+---------+----------+---------+---------+---------+----------+---------+----------+---------+---------------+
|           all-mpnet-base-v2           |  0.7800 |  0.7785  |  0.7644 |  0.8574 |  0.5100 |  0.8531  |  0.8650 |  0.9149  |  0.8190 |      1.81     |
|       multi-qa-mpnet-base-dot-v1      |  0.7800 |  0.7667  |  0.7605 |  0.8482 |  0.5079 |  0.8538  |  0.7150 |  0.6600  |  0.9429 |      1.78     |
|          all-distilroberta-v1         |  0.8000 |  0.7987  |  0.7854 |  0.8434 |  0.5018 |  0.8496  |  0.8200 |  0.8710  |  0.7714 |      0.95     |
|           all-MiniLM-L12-v2           |  0.8300 |  0.8383  |  0.8156 | 

In [13]:
from PIL import Image, ImageDraw, ImageFont

In [34]:
im = Image.new("RGB", (1500, 300), "white")
draw = ImageDraw.Draw(im)
font = ImageFont.truetype(r"D:\Downloads\freemono\FreeMonoBold.ttf", 15)
draw.text((10, 10), str(table), font=font, fill="black")

im.show()
im.save("table.png")